# Data Generation

## libraries

In [13]:
# =============================================
# Core Scientific/LCA (Life Cycle Assessment) Libraries
# Required for Brightway2.5 (LCA framework)
# Always place these at the top, as they are the main dependencies for your project
# =============================================
import os
import bw2io as bi # Brightway2 library for data import/export
import bw2data as bd # Core Brightway2 library for LCA database management
import bw2calc as bc

# =============================================
# Uncertainty and Statistical Distributions
# Used for handling uncertainty in LCA data
# Grouped together for clarity
# =============================================
from stats_arrays.distributions import (
    LognormalUncertainty,  # For log-normal uncertainty distributions
    NormalUncertainty,     # For normal uncertainty distributions
    TriangularUncertainty, # For triangular uncertainty distributions
    UniformUncertainty,     # For uniform uncertainty distributions
    UndefinedUncertainty,  # For undefined uncertainty distributions
)

# =============================================
# Data Processing and Numerical Libraries
# Used for data manipulation, math, and array operations
# =============================================
import numpy as np  # Numerical computing (arrays, math operations), only version for np.nan # numpy==1.26.4
import pandas as pd # Data manipulation and analysis (DataFrames)
import math         # Basic math functions
import csv          # CSV file reading/writing
import ast          # Abstract Syntax Trees (for parsing Python literals)
import json         # JSON data handling (should be with other data processing libraries)
import os           # Operating system interfaces (file paths, environment variables)

# =============================================
# File System and Path Handling
# Used for file and directory operations
# =============================================
from pathlib import Path  # Object-oriented filesystem paths

# =============================================
# Time and Delay Utilities
# Used for introducing delays or measuring time
# =============================================
from time import sleep  # Introduce delays (e.g., for web scraping)
import time             # Time-related functions (e.g., `time.time()`)

# =============================================
# Chemistry and Data Retrieval
# Used for chemical data and external API access
# =============================================
import pubchempy as pcp  # Interface to PubChem (chemical data)
import requests         # HTTP requests (e.g., API calls)
from mendeleev import element  # Periodic table data (element properties)

# =============================================
# XML/HTML Parsing
# Used for parsing XML/HTML files
# =============================================
from lxml import etree  # XML/HTML parsing
import html             # HTML escaping/unescaping
import re               # Regular expressions (text pattern matching)

# =============================================
# Data Structures
# Used for advanced data structures
# =============================================
from collections import defaultdict  # Default dictionary (for grouping data)

## functions

In [14]:
# Functions for TheRy

# 1. The name of each flow will be cleaned using re
def clean_chemical_name(name):
    # This detects some of the organic molecules (e.g. Ethane, 1,1-difluoro-, HFC-152a)
    pattern1 = r"^(.+?),\s+(.+?)-(?:,\s+.*|$)"
    # This pattern simple detects the flows with a comma.
    pattern2 = r"(?<=[a-zA-Z]),.*"

    # First, check if the flow satisfies pattern1
    match = re.search(pattern1,name)
    # If there is a match, then we change its format (e.g. from Ethane, 1,1-difluoro-, HFC-152a to 1,1-difluoroEthane)
    if match:
        name = re.sub(pattern1, r"\g<2>\g<1>", name)
    # If not, we apply the 2nd pattern
    else:
        name = re.sub(pattern2, "", name)
    
    # Second, remove Roman numerals at the end (e.g., Aluminium III -> Aluminium) if applicable
    name = re.sub(r"\s+[IVXLCDM]+$", "", name)
    
    # 3. Third the word 'ion' or 'ions' at the end (e.g., Copper ion -> Copper) if applicable
    # Using re.IGNORECASE makes it catch 'Ion', 'ION', or 'ion'
    name = re.sub(r"\s+ions?$", "", name, flags=re.IGNORECASE)
    
    return name.strip()


# 2. Then, it will be fed to the API to find the chemical. If it is found, the original flow name, the chemical IUPAC name, molecular composition, and the molecular weight will be made into a dictionary
def chem_comp(mf):
    # First of all, we need to use re to cut the molecular formula into pieces
    # Regex breakdown:
    # ([A-Z][a-z]?) -> Matches an Uppercase letter potentially followed by a lowercase (the element)
    # (\d*)         -> Matches zero or more digits following the element (the count)
    pattern = r"([A-Z][a-z]?)(\d*)"
    
    matches = re.findall(pattern, mf)
    
    parts = []

    for elem,count in matches:
        # If the count is empty (like in 'Cl'), it means there is 1 atom
        count = int(count) if count else 1
        parts.append([elem,count])
    
    return parts

def has_metal(parts):
    for elem, count in parts:
        if elem not in valero_rarity_data.keys(): #if it is not a metal
            print(f"{elem} is not a metal.")
            continue
        else:
            return True
    return None

# 3. Calculates for the CF of the given molecular formular
def cf_calculator(mf,mw,parts):
    cf=0 # place holder for the 

    for elem, count in parts:
        if elem in valero_rarity_data.keys():
            # Find the atomic mass of the element first
            el_data = element(elem) # from mendeley ?
            mass = float(el_data.atomic_weight) # from mendeley ?
            # Compute for the contribution of this element in the final mf
            cf += count*(mass/mw)*valero_rarity_data[elem]
        else:
            continue

    return round(cf,3)

## data scraping and method creation

In [15]:
# New project
project_name = "CERC"
bd.projects.set_current(project_name)

# Récupère les identifiants depuis les variables d'environnement
username = os.getenv("EI_USERNAME")
password = os.getenv("EI_PASSWORD")

# Utilise-les pour importer ecoinvent
#bi.import_ecoinvent_release("3.4", "cutoff", username, password)

In [16]:
bd.databases


Databases dictionary with 2 object(s):
	ecoinvent-3.4-biosphere
	ecoinvent-3.4-cutoff

In [27]:
import pandas as pd
import re

# First, let's read the CSV file
valero_df = pd.read_csv("C:/Users/louis/github/TheRy-Index/valero-constants.csv")

# Count non-null ERC values in the dataframe
elements_with_erc = valero_df['ERC[MJ/kg]'].notna().sum()

# Clean the column names (they seem to have some encoding issues)
valero_df.columns = ['name', 'k_x_xm', 'MW', 'xm_g_g', 'ERC[MJ/kg]']

# Manual mapping from the CSV names to element symbols
# This is needed because the CSV contains mineral names, not element names
name_to_symbol = {
    "Aluminium - Bauxite (Gibbsite)": "Al",
    "Antimony (Stibnite)": "Sb",
    "Arsenic (Arsenopyrite)": "As",
    "Barite": "Ba",
    "Beryllium (Beryl)": "Be",
    "Bismuth (Bismuthinite)": "Bi",
    "Cadmium (Greenockite)": "Cd",
    "Cerium (Monazite)": "Ce",
    "Chromium (Chromite)": "Cr",
    "Cobalt (Linnaeite)": "Co",
    "Copper (Chalcopyrite)": "Cu",
    "Fluorite": "F",
    "Gadolinium-Monazite": "Gd",
    "Gallium (in Bauxite)": "Ga",
    "Germanium (in Zinc)": "Ge",
    "Gold": "Au",
    "Graphite": "C",
    "Gypsum": "Ca",  # Gypsum is CaSO4·2H2O
    "Hafnium": "Hf",
    "Indium (in Zinc)": "In",
    "Iron ore (Hematite)": "Fe",
    "Lanthanum-Monazite": "La",
    "Lead (Galena)": "Pb",
    "Lime": "Ca",  # Lime is CaO
    "Lithium (Spodumene)": "Li",
    "Magnesite (from ocean)": "Mg",
    "Manganese (Pyrolusite)": "Mn",
    "Mercury (Cinnabar)": "Hg",
    "Molybdenum (Molybdenite)": "Mo",
    "Neodymium-Monazite": "Nd",
    "Nickel (sulphides) Pentlandite": "Ni",
    "Nickel (laterites) Garnierite": "Ni",
    "Niobium (ferrocolumbite)": "Nb",
    "Palladium": "Pd",
    "Phosphate rock (Apatite)": "P",
    "PGM (average value for all PGM)": "PGMs (average)",
    "Platinum": "Pt",
    "Potassium (Sylvite)": "K",
    "Praseodymium-Monazite": "Pr",
    "REE (Bastnaesite)": "REE",
    "Rhenium": "Re",
    "Selenium": "Se",
    "Silicon (Quartz)": "Si",
    "Silver (Argentite)": "Ag",
    "Sodium (Halite)": "Na",
    "Strontium": "Sr",
    "Tantalum (Tantalite)": "Ta",
    "Tellurium-Tetradymite": "Te",
    "Thallium": "Tl",
    "Tin (Cassiterite)": "Sn",
    "Titanium (Ilmenite)": "Ti",
    "Titanium (Rutile)": "Ti",
    "Uranium (Uraninite)": "U",
    "Vanadium": "V",
    "Wolfram (Scheelite)": "W",
    "Yttrium-Monazite": "Y",
    "Zinc (Sphalerite)": "Zn",
    "Zirconium (Zircon)": "Zr"
}

# Create the new thermodynamic rarity dictionary from valero data
valero_rarity_data = {}

for index, row in valero_df.iterrows():
    name = row['name']
    erc_value = row['ERC[MJ/kg]']
    
    if pd.notna(erc_value) and name in name_to_symbol:
        symbol = name_to_symbol[name]
        # Convert scientific notation to float
        valero_rarity_data[symbol] = float(erc_value)

# Now add the missing REE and PGM elements using the group averages
ree_elements = ["Sc", "Y", "La", "Ce", "Pr", "Nd", "Pm", "Sm", "Eu", "Gd", "Tb", "Dy", "Ho", "Er", "Tm", "Yb", "Lu"]
pgm_elements = ["Pt", "Pd", "Rh", "Ru", "Ir", "Os"]

# Fill in missing REE elements with the REE average
if "REE" in valero_rarity_data:
    for elem in ree_elements:
        if elem not in valero_rarity_data:
            valero_rarity_data[elem] = valero_rarity_data["REE"]
            print(f"Added {elem} using REE average value: {valero_rarity_data['REE']}")

# Fill in missing PGM elements with the PGM average
if "PGMs (average)" in valero_rarity_data:
    for elem in pgm_elements:
        if elem not in valero_rarity_data:
            valero_rarity_data[elem] = valero_rarity_data["PGMs (average)"]
            print(f"Added {elem} using PGM average value: {valero_rarity_data['PGMs (average)']}")

print(f"\nTotal elements in valero-based dictionary: {len(valero_rarity_data)}")
print("\nValero-based thermodynamic rarity data:")
for symbol, value in sorted(valero_rarity_data.items()):
    print(f'"{symbol}": {value},')

# Count elements in the valero rarity dictionary
elements_in_valero_rarity_data = len(valero_rarity_data)

Added Sc using REE average value: 348.0
Added Pm using REE average value: 348.0
Added Sm using REE average value: 348.0
Added Eu using REE average value: 348.0
Added Tb using REE average value: 348.0
Added Dy using REE average value: 348.0
Added Ho using REE average value: 348.0
Added Er using REE average value: 348.0
Added Tm using REE average value: 348.0
Added Yb using REE average value: 348.0
Added Lu using REE average value: 348.0
Added Rh using PGM average value: 2700000.0
Added Ru using PGM average value: 2700000.0
Added Ir using PGM average value: 2700000.0
Added Os using PGM average value: 2700000.0

Total elements in valero-based dictionary: 70

Valero-based thermodynamic rarity data:
"Ag": 7370.0,
"Al": 627.0,
"As": 400.0,
"Au": 553000.0,
"Ba": 38.0,
"Be": 253.0,
"Bi": 489.0,
"C": 20.0,
"Ca": 3.0,
"Cd": 5900.0,
"Ce": 97.0,
"Co": 10900.0,
"Cr": 5.0,
"Cu": 292.0,
"Dy": 348.0,
"Er": 348.0,
"Eu": 348.0,
"F": 183.0,
"Fe": 18.0,
"Ga": 145000.0,
"Gd": 478.0,
"Ge": 23700.0,
"Hf": 21

In [18]:
# 1. Initialize the database object
biosphere = bd.Database('ecoinvent-3.4-biosphere')

# 2. Extract data into a list of dictionaries
# We use .as_dict() to get all the metadata for each flow
data = []
for flow in biosphere:
    flow_dict = flow.as_dict()
    
    # Flatten categories (e.g., ('air', 'urban') -> 'air, urban')
    if 'categories' in flow_dict:
        flow_dict['categories'] = ", ".join(flow_dict['categories'])
    
    data.append(flow_dict)

# 3. Create the DataFrame
df = pd.DataFrame(data)

# 4. Save to CSV
df.to_csv("data/EI_biosphere_flows.csv", index=False)
print("✅ Exported biosphere to biosphere_flows.csv")

✅ Exported biosphere to biosphere_flows.csv


In [19]:
import ssl

# This overrides the default SSL behavior for the current session
ssl._create_default_https_context = ssl._create_unverified_context

In [20]:
from pubchempy import PubChemHTTPError

# we need a if-else filter here to see if the flow is elementary. If yes, do not go through the cf_calculator

method_data = []
extra_info = []

not_found = []
not_emission = [] # can be used to check for the flows that aren't considered

i=0
for flow in biosphere:
    cf = None
    parts = None #reset parts every time
    # Only consider the elementary flows going into the biosphere
    # Find flows that contain any of the search terms and don't have 'natural resource' as the first category
    if isinstance(flow.get('categories'), tuple) and len(flow['categories']) > 0 and flow['unit'].lower() == 'kilogram' and flow['categories'][0].lower() != 'natural resource':

        flow_name_cleaned = clean_chemical_name(flow['name'])

        sleep(0.3) #Slow down the request to the API to avoid sudden shut down of connection

        #NEW EDIT: added a try,except loop (26 June 2026); worked like a charm! 
        max_retries = 3
        for attempt in range(max_retries):
            try:
                compound = pcp.get_compounds(flow_name_cleaned,'name')
                if compound: 
                    c = compound[0]
                    mf = c.molecular_formula
                    mw = c.molecular_weight
            
                    parts = chem_comp(mf)
            

                    # If the flow contains only one element, we can directly use its CF value
                    if len(parts)==1:
                        if has_metal(parts): #if the only element is a part of the metal group
                            cf = valero_rarity_data.get(parts[0][0])   
                            print(f"The cf of {mf} is decided to be {cf}")
                        else:
                            print(f"{flow['name']} does not contain any metal.")       
                    # If the flow contains a compound, use the cf_calculator
                    elif len(parts)>1:
                        if has_metal(parts):
                            cf = cf_calculator(mf,mw,parts)
                            print(f"The cf of {mf} is calculated to be {cf}")
                        else:
                            print(f"{flow['name']} does not contain any metal.")   
                    else: 
                        print(f"The molecular compound does not contain any element.")
                    
                    # Combine the original flow name, 
                    if cf is not None:
                        method_data.append((flow.key,float(cf)))
                        extra_info.append([flow.key,flow['name'],flow_name_cleaned,mf,mw,float(cf)])
                    else: 
                        extra_info.append([flow.key,flow['name'],flow_name_cleaned,mf,mw,None])
                        print('This compound does not contain any metal.')
                else:
                    print(f"Compound not found for: {flow['name']}")
                    not_found.append([flow.key,flow['name']])
                break
            
            except PubChemHTTPError as e:
                # Check if it's a 502 or another server error
                if "502" in str(e) and attempt < max_retries - 1:
                    print(f"Server hit a 502 for {flow_name_cleaned}. Retrying in 2 seconds... (Attempt {attempt + 1}/{max_retries})")
                    time.sleep(2)  # Give the server a moment to recover
                else:
                    print(f"Failed to fetch {flow_name_cleaned} after multiple attempts or met a different error: {e}")
                    # Handle or log the permanent failure here
                    break
    else:
        not_emission.append([flow.key,flow['name']])


The cf of C6H14 is calculated to be 16.725
The cf of Se is decided to be 2240000.0
The cf of C2H4O is calculated to be 10.907
The cf of F- is decided to be 183.0
The cf of Be is decided to be 253.0
The cf of C12H21N2O3PS is calculated to be 9.507
The cf of C16H16N4O2 is calculated to be 12.971
The cf of Ag is decided to be 7370.0
Compound not found for: Hydrocarbons, unspecified
The cf of C16H22ClN3O is calculated to be 12.486
The cf of C3H9N is calculated to be 12.192
The cf of C2H7N is calculated to be 10.657
The cf of C19H18ClN3O4 is calculated to be 11.769
The cf of Co is decided to be 10900.0
The cf of C6H6 is calculated to be 18.452
Cl is not a metal.
Chloride does not contain any metal.
This compound does not contain any metal.
The cf of C4H6O2 is calculated to be 11.161
The cf of C3H9N is calculated to be 12.192
The cf of Ni is decided to be 167.0
The cf of Ni is decided to be 167.0
Compound not found for: Solids, inorganic
Server hit a 502 for Fluometuron. Retrying in 2 second

In [22]:
# Export the data scraped from the API to a csv file
PC_df = pd.DataFrame(method_data)
PC_df.to_csv("results/PubChem_ERC_EI34.csv", index=False)

In [23]:
df = pd.read_csv('results/PubChem_ERC_EI34.csv')
unique_flows = df['1'].unique()
len(unique_flows)

521

In [28]:
PC_df.shape

(2438, 2)

In [30]:
PC_df = pd.DataFrame(extra_info)
NF_df = pd.DataFrame(not_found)
NInc_df = pd.DataFrame(not_emission)
PC_df.to_csv("results/PubChem_Info_EI34.csv", index=False)
NF_df.to_csv("results/PubChem_NotFound_EI34.csv", index=False)
NInc_df.to_csv("results/PubChem_NotEmission_EI34.csv", index=False)

In [31]:
# 1. Define the method name as a tuple for the desired hierarchy
# Define the method name as a tuple for the desired hierarchy
method_name_tuple = (
    "Cumulative Exergy Replacement Cost",
    "Dissipation-based",
    f"ERC total: To Repurpose all dissipated flows as resources applied to {len(valero_rarity_data)}"
)

# 2. Define metadata for the method, including the unit and a description
method_metadata = {
    'unit': 'MJ-Eq',
    'description': 'Exergy Replacement Cost applied to Output Product-System Metals within Elementary flows. '
                  'The Characterization factors represent the exergy (MJ/kg) from nature and'
                  'current human systems used to make 1 kg of the resource available. '
                  'Applied to the output flows, it is assumed that all these output flows reach an environment compartment with the Thanatia concentration.'
                  'Therefore, the product-system in focus implies the future use of exergy for future other product-systems that will seek these resources.',
    'source': 'The values are taken from Alicia Valero works and can be found in the file valero-constants.csv',
    'version': '1.0', 
    'num_cfs': len(method_data),
    'application': 'Output product-system metals characterization'
}

# 3. Create or load the Brightway2 Method object
method_object = bd.Method(method_name_tuple)

# 4. Forcefully register/write the method
try:
    # This will overwrite existing method
    method_object.register(**method_metadata) 
    method_object.write(method_data)
    
    print(f"\n✅ Successfully {'overwrote' if method_name_tuple in bd.methods else 'created'} method: {method_name_tuple}")
    print(f"   - Unit: {method_metadata['unit']}")
    print(f"   - Number of CFs: {len(method_data)}")
    
except Exception as e:
    print(f"\n❌ Failed to write method: {str(e)}")
    raise


# --- Verification ---
print("\n--- Verification ---")
if method_name_tuple in bd.methods:
    retrieved_method = bd.Method(method_name_tuple)
    loaded_data = retrieved_method.load()
    
    print(f"🔍 Method verification:")
    print(f"   Name: {retrieved_method.name}")
    print(f"   Metadata version: {retrieved_method.metadata.get('version', 'N/A')}")
    print(f"   Number of CFs loaded: {len(loaded_data)}")
    
    # Check for potential data loss
    if len(loaded_data) != len(method_data):
        print(f"⚠️  Warning: CF count mismatch. Expected {len(method_data)}, got {len(loaded_data)}")
    else:
        print("✅ CF count matches expected value")
        
    # Show sample CFs
    print("\nSample characterization factors (first 3):")
    for cf in loaded_data[:3]:
        flow = bd.get_activity(cf[0])
        print(f"   - {flow['name']}: {cf[1]} MJ-Eq")
else:
    print(f"❌ Error: Method {method_name_tuple} not found after writing attempt")

print("\nProcess completed")


✅ Successfully overwrote method: ('Cumulative Exergy Replacement Cost', 'Dissipation-based', 'ERC total: To Repurpose all dissipated flows as resources applied to 70')
   - Unit: MJ-Eq
   - Number of CFs: 2438

--- Verification ---
🔍 Method verification:
   Name: ('Cumulative Exergy Replacement Cost', 'Dissipation-based', 'ERC total: To Repurpose all dissipated flows as resources applied to 70')
   Metadata version: 1.0
   Number of CFs loaded: 2438
✅ CF count matches expected value

Sample characterization factors (first 3):
   - 2-Methyl pentane: 16.725 MJ-Eq
   - Selenium: 2240000.0 MJ-Eq
   - Ethylene oxide: 10.907 MJ-Eq

Process completed


# Flow Analysis with ERC

In [32]:
print(f"Current project: {bd.projects.current}")

# List ALL methods in this project to see if yours is there
for m in bd.methods:
    print(m)

Current project: CERC
('ecoinvent-3.4', 'CML 2001', 'acidification potential', 'average European')
('ecoinvent-3.4', 'CML 2001', 'acidification potential', 'generic')
('ecoinvent-3.4', 'CML 2001', 'climate change', 'GWP 100a')
('ecoinvent-3.4', 'CML 2001', 'climate change', 'GWP 20a')
('ecoinvent-3.4', 'CML 2001', 'climate change', 'GWP 500a')
('ecoinvent-3.4', 'CML 2001', 'climate change', 'lower limit of net GWP')
('ecoinvent-3.4', 'CML 2001', 'climate change', 'upper limit of net GWP')
('ecoinvent-3.4', 'CML 2001', 'eutrophication potential', 'average European')
('ecoinvent-3.4', 'CML 2001', 'eutrophication potential', 'generic')
('ecoinvent-3.4', 'CML 2001', 'freshwater aquatic ecotoxicity', 'FAETP 100a')
('ecoinvent-3.4', 'CML 2001', 'freshwater aquatic ecotoxicity', 'FAETP 20a')
('ecoinvent-3.4', 'CML 2001', 'freshwater aquatic ecotoxicity', 'FAETP 500a')
('ecoinvent-3.4', 'CML 2001', 'freshwater aquatic ecotoxicity', 'FAETP infinite')
('ecoinvent-3.4', 'CML 2001', 'freshwater se

In [33]:
my_db = bd.Database("ecoinvent-3.4-cutoff")
results = my_db.search("iron production")

# 3. Look at the results
for activity in results:
    print(f"Region: {activity['location']}, Name: {activity['name']}, Code: {activity['code']}")

Region: GLO, Name: market for iron(III) chloride, without water, in 14% iron solution state, Code: a54978781257bff94613c03f3d2780fa
Region: GLO, Name: market for iron(III) sulfate, without water, in 12.5% iron solution state, Code: 26822caa5afb16a5003cebd655562926
Region: GLO, Name: market for iron(III) chloride, without water, in a 12% iron solution state, Code: 5e8d251ef05373bd5e94ade8cf92e9c7
Region: RoW, Name: market for wastewater from pig iron production, Code: c40a282af7cd853cfdf411b123285267
Region: CA-QC, Name: iron(III) chloride production, without water, in 14% iron solution state, Code: 6aab39ddd795c3c17495caa45e8e6252
Region: CA-QC, Name: iron(III) chloride production, without water, in 12% iron solution state, Code: aa1548acec01e9d9a6111f27a9273e53
Region: RoW, Name: iron(III) chloride production, without water, in 14% iron solution state, Code: 6df9490e5a577ae73d7294fe43d55d57
Region: RoW, Name: iron(III) chloride production, without water, in 12% iron solution state, Co

### GLO pig iron production

In [34]:
# Setup your functional unit and method
my_act = bd.Database("ecoinvent-3.4-cutoff").get("18107bc7b3e28650e3fd8cf16da9aed9")

lca = bc.LCA({my_act: 1}, method_name_tuple)
lca.lci()
lca.lcia()
print(f"Total Impact: {lca.score}")

Total Impact: 17.566405270029904


In [35]:
# Setup your functional unit and method
my_act = bd.Database("ecoinvent-3.4-cutoff").get("18107bc7b3e28650e3fd8cf16da9aed9")
lca = bc.LCA({my_act: 1}, method_name_tuple)
lca.lci()
lca.lcia()
print(f"Total Impact: {lca.score}")

Total Impact: 17.566405270029904


In [36]:
# Setup your functional unit and method
my_act = bd.Database("ecoinvent-3.4-cutoff").get("18107bc7b3e28650e3fd8cf16da9aed9")
method = ('ecoinvent-3.4', 'cumulative exergy demand', 'minerals', 'non-renewable material resources, minerals')

lca = bc.LCA({my_act: 1}, method)
lca.lci()
lca.lcia()
print(f"Total Impact: {lca.score}")

Total Impact: 0.01485416359130504
